In [ ]:
import os
import glob
from pathlib import Path
import numpy as np
import scipy.io
from scipy.fft import rfft, rfftfreq
import matplotlib.pyplot as plt

# Configuração de estilo sóbrio conforme padrão ABNT / periódicos científicos
plt.rcParams.update({
    'font.size': 16,
    'font.family': 'serif',
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 12,
    'lines.linewidth': 0.8,
    'grid.alpha': 0.5,
    'grid.linestyle': '--'
})

In [ ]:
# Definição do ensaio de interesse (aceita "12", 12 ou "B12")
exp = "B17"
exp_id = exp.upper() if str(exp).upper().startswith("B") else f"B{int(exp):02d}"

# Resolução determinística do diretório-raiz do projeto
caminho_atual = Path(os.getcwd())
raiz_projeto = caminho_atual.parent if caminho_atual.name == "notebooks" else caminho_atual

# Estruturação do caminho dos dados brutos de vibração
pasta_exp = raiz_projeto / "data" / "raw_data" / exp_id / "vibrationData"
print(f"Diretório dos dados ({exp_id}): {pasta_exp.resolve()}")

# Varredura determinística de arquivos .mat conforme o padrão de nomenclatura
arquivos_mat = sorted(pasta_exp.glob(f"data_{exp_id}_M*.mat"))
if not arquivos_mat:
    arquivos_mat = sorted(pasta_exp.glob("*.mat"))

if not arquivos_mat:
    raise FileNotFoundError(f"Nenhum arquivo .mat localizado no diretório: {pasta_exp}")

# Seleção dos marcos temporais (início saudável e condição final de falha funcional)
path_saudavel = arquivos_mat[0]
path_falha = arquivos_mat[-5]

print(f"Total de medições encontradas ({exp_id}): {len(arquivos_mat)}")
print(f"Condição saudável inicial: {path_saudavel.name}")
print(f"Falha funcional final:    {path_falha.name}")

In [ ]:
def carregar_sinal_paderborn(caminho_arquivo, canal='accHorizFrontal_C', sensibilidade=0.1):
    """
    Extrai o sinal de vibração do arquivo .mat (Aimiyekagbon, 2024), remove o
    offset DC (Wei et al., 2025) e converte a amplitude de Volts para unidades
    de aceleração gravitacional [g].

    Parâmetros:
    -----------
    caminho_arquivo : str ou Path
        Caminho para o arquivo .mat da medição.
    canal : str, padrão='accHorizFrontal_C'
        Chave do acelerômetro ('accHorizFrontal_C' ou 'accHorizRear_A').
    sensibilidade : float, padrão=0.1
        Sensibilidade nominal do sensor PCB 603C01 em V/g (100 mV/g).

    Retorno:
    --------
    sinal_g : np.ndarray
        Vetor unidimensional de aceleração centrado e calibrado em [g].
    """
    dados = scipy.io.loadmat(caminho_arquivo)
    
    if canal in dados:
        sinal = dados[canal].flatten()
    else:
        chaves_validas = [k for k in dados.keys() if not k.startswith('__')]
        sinal = dados[chaves_validas[0]].flatten()
        
    # Remoção do componente de corrente contínua (offset DC de instrumentação)
    sinal_centrado = sinal - np.mean(sinal)
    
    # Conversão de escala: Volts [V] para aceleração gravitacional [g]
    sinal_g = sinal_centrado / sensibilidade
    return sinal_g


def calcular_espectro_amplitude(sinal, fs=64000):
    """
    Computa a FFT unilateral com janelamento de Hanning e restauração de amplitude.
    
    Parâmetros:
    -----------
    sinal : np.ndarray
        Vetor de aceleração calibrado em [g].
    fs : int, padrão=64000
        Frequência de amostragem em Hz.

    Retorno:
    --------
    frequencias : np.ndarray
        Vetor de frequências unilaterais em [Hz].
    espectro_amplitude : np.ndarray
        Espectro de amplitude calibrado na mesma unidade de entrada [g].
    """
    n_pontos = len(sinal)
    janela = np.hanning(n_pontos)
    sinal_janelado = sinal * janela
    
    frequencias = rfftfreq(n_pontos, d=1.0/fs)
    fft_valores = rfft(sinal_janelado)
    
    # Fator de correção de amplitude para a janela de Hanning: 2 / sum(janela)
    espectro_amplitude = (2.0 / np.sum(janela)) * np.abs(fft_valores)
    return frequencias, espectro_amplitude

In [ ]:
FS = 64000  # Frequência de amostragem padronizada em 64 kHz (Aimiyekagbon, 2024)
CANAL = 'accHorizFrontal_C'  # Sensor horizontal frontal C

# Carregamento dos vetores de processo
sinal_saudavel = carregar_sinal_paderborn(path_saudavel, canal=CANAL)
sinal_falha = carregar_sinal_paderborn(path_falha, canal=CANAL)

# Vetores da escala temporal
tempo_saudavel = np.arange(len(sinal_saudavel)) / FS
tempo_falha = np.arange(len(sinal_falha)) / FS

# Transformada de Fourier unilateral
freq_saudavel, amp_saudavel = calcular_espectro_amplitude(sinal_saudavel, fs=FS)
freq_falha, amp_falha = calcular_espectro_amplitude(sinal_falha, fs=FS)

print("Processamento de dados concluído com sucesso.")

In [ ]:
# -------------------------------------------------------------------------
# Sinal de Vibração no Domínio do Tempo – Ensaio {exp_id}
# -------------------------------------------------------------------------
t_lim = 0.05  # Janela de análise de 50 ms para visualização de transitórios

idx_t_h = tempo_saudavel <= t_lim
idx_t_m = tempo_falha <= t_lim

# Determinação do limite simétrico da escala vertical de amplitude (V)
max_amp_t = max(np.max(np.abs(sinal_saudavel[idx_t_h])), 
                np.max(np.abs(sinal_falha[idx_t_m]))) * 1.1

fig_tempo, axs_t = plt.subplots(2, 1, figsize=(7, 4), sharex=True, sharey=True, constrained_layout=True)

# (a) Início do ensaio (Condição saudável)
axs_t[0].plot(tempo_saudavel[idx_t_h], sinal_saudavel[idx_t_h], color='#1f77b4')
axs_t[0].set_title("(a)", loc="left", fontsize=10, weight="bold")
axs_t[0].set_ylabel("Amplitude (g)")
axs_t[0].grid(True)

# (b) Estágio avançado/falha do ensaio
axs_t[1].plot(tempo_falha[idx_t_m], sinal_falha[idx_t_m], color='#d62728')
axs_t[1].set_title("(b)", loc="left", fontsize=10, weight="bold")
axs_t[1].set_xlabel("Tempo (s)")
axs_t[1].set_ylabel("Amplitude (g)")
axs_t[1].grid(True)

# Equalização dos eixos
axs_t[0].set_xlim(0, t_lim)
axs_t[0].set_ylim(-max_amp_t, max_amp_t)

plt.show()

In [ ]:
# -------------------------------------------------------------------------
# Espectro de Amplitude Unilateral (FFT) – Ensaio {exp_id}
# -------------------------------------------------------------------------
f_lim = 10000  # Banda de interesse: 0 a 10 kHz

idx_f_h = freq_saudavel <= f_lim
idx_f_m = freq_falha <= f_lim

# Determinação do limite superior da escala de amplitude espectral (V)
max_amp_f = max(np.max(amp_saudavel[idx_f_h]), 
                np.max(amp_falha[idx_f_m])) * 1.05

fig_freq, axs_f = plt.subplots(2, 1, figsize=(7, 4), sharex=True, sharey=True, constrained_layout=True)

# (a) Condição inicial saudável
axs_f[0].plot(freq_saudavel[idx_f_h], amp_saudavel[idx_f_h], color='#1f77b4', linewidth=1)
axs_f[0].set_title('(a)', loc='left', fontsize=10, weight='bold')
axs_f[0].set_ylabel('Amplitude (g)')
axs_f[0].grid(True, linestyle='--', alpha=0.6)

# (b) Estágio avançado de degradação / falha funcional
axs_f[1].plot(freq_falha[idx_f_m], amp_falha[idx_f_m], color='#d62728', linewidth=1)
axs_f[1].set_title('(b)', loc='left', fontsize=10, weight='bold')
axs_f[1].set_xlabel('Frequência (Hz)')
axs_f[1].set_ylabel('Amplitude (g)')
axs_f[1].grid(True, linestyle='--', alpha=0.6)

# Equalização exata das escalas espectrais
axs_f[0].set_xlim(0, f_lim)
axs_f[0].set_ylim(0, max_amp_f)

plt.show()

In [ ]:
import scipy.signal
from scipy.fft import fft, ifft, fftfreq


def calcular_espectrograma_stft(sinal, fs=64000, nperseg=256, noverlap=240, nfft=1024):
    """
    Calcula o espectrograma de amplitude unilateral via Transformada de Fourier
    de Tempo Curto (STFT) com janela de Hanning.

    Parâmetros:
    -----------
    sinal : np.ndarray
        Vetor de aceleração calibrado em [g].
    fs : int, padrão=64000
        Frequência de amostragem em Hz.
    nperseg : int, padrão=256
        Comprimento da janela móvel em amostras (256 amostras = 4,0 ms a 64 kHz).
    noverlap : int, padrão=240
        Sobreposição entre janelas consecutivas em amostras (passo de 0,25 ms).
    nfft : int, padrão=1024
        Número de pontos da FFT por janela (interpolação espectral de 62,5 Hz).

    Retorno:
    --------
    tempos : np.ndarray
        Vetor de instantes centrais das janelas em [s].
    frequencias : np.ndarray
        Vetor de frequências unilaterais em [Hz].
    amplitude : np.ndarray
        Matriz tempo-frequência de amplitude calibrada em [g].
    """
    frequencias, tempos, Zxx = scipy.signal.stft(
        sinal,
        fs=fs,
        window='hann',
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
        scaling='spectrum',
        boundary='even',
        padded=True
    )

    amplitude = np.abs(Zxx)
    # Fator unilateral para amplitude de pico [g] (exclui DC e Nyquist)
    if nfft % 2 == 0:
        amplitude[1:-1, :] *= 2.0
    else:
        amplitude[1:, :] *= 2.0

    return tempos, frequencias, amplitude


def calcular_escalograma_morlet(sinal, fs=64000, f_min=100.0, f_max=10000.0,
                                n_freqs=200, omega0=6.0):
    """
    Calcula o escalograma via Transformada Wavelet Contínua (CWT) com wavelet
    analítica de Morlet no domínio da frequência (normalização L1 em [g]).

    Parâmetros:
    -----------
    sinal : np.ndarray
        Vetor de aceleração calibrado em [g].
    fs : int, padrão=64000
        Frequência de amostragem em Hz.
    f_min : float, padrão=100.0
        Frequência mínima da grade de análise em Hz.
    f_max : float, padrão=10000.0
        Frequência máxima da grade de análise em Hz.
    n_freqs : int, padrão=200
        Número de frequências na grade linear para alinhamento visual com a STFT.
    omega0 : float, padrão=6.0
        Frequência central adimensional da wavelet-mãe de Morlet.

    Retorno:
    --------
    tempos : np.ndarray
        Vetor de instantes temporais em [s].
    frequencias : np.ndarray
        Vetor de pseudo-frequências em [Hz].
    escalograma : np.ndarray
        Matriz tempo-frequência de magnitude calibrada em [g].
    """
    n_orig = len(sinal)
    # Preenchimento simétrico para atenuar efeitos de borda na convolução via FFT
    pad = min(n_orig // 2, int(0.02 * fs))
    sinal_pad = np.pad(sinal, (pad, pad), mode='reflect')
    n_pad = len(sinal_pad)

    freqs_fft = fftfreq(n_pad, d=1.0 / fs)
    fft_sinal = fft(sinal_pad)

    frequencias = np.linspace(f_min, f_max, n_freqs)
    escalograma = np.empty((n_freqs, n_orig), dtype=np.float64)

    mascara_pos = freqs_fft > 0.0
    f_pos = freqs_fft[mascara_pos]

    for idx, f_alvo in enumerate(frequencias):
        # Resposta em frequência da wavelet analítica de Morlet (normalização L1)
        psi_hat = np.zeros(n_pad, dtype=np.float64)
        psi_hat[mascara_pos] = 2.0 * np.exp(-0.5 * (omega0 * (f_pos / f_alvo - 1.0)) ** 2)

        coef_complexo = ifft(fft_sinal * psi_hat)
        escalograma[idx, :] = np.abs(coef_complexo[pad:pad + n_orig])

    tempos = np.arange(n_orig) / fs
    return tempos, frequencias, escalograma

In [ ]:
# -------------------------------------------------------------------------
# Representações Tempo-Frequência (STFT e CWT) – Ensaio {exp_id} (Falha)
# -------------------------------------------------------------------------
t_lim_tf = 0.05   # Mesma janela de 50 ms da Figura 2
f_lim_tf = 10000  # Mesma banda de 0 a 10 kHz da Figura 3 (faixa útil do sensor)

idx_tf = tempo_falha <= t_lim_tf
sinal_trecho = sinal_falha[idx_tf]

# Cálculo do espectrograma (STFT) e do escalograma (CWT de Morlet)
t_stft, f_stft, amp_stft = calcular_espectrograma_stft(
    sinal_trecho, fs=FS, nperseg=256, noverlap=240, nfft=1024
)
t_cwt, f_cwt, amp_cwt = calcular_escalograma_morlet(
    sinal_trecho, fs=FS, f_min=100.0, f_max=f_lim_tf, n_freqs=200, omega0=6.0
)

# Recorte espectral da STFT na banda de interesse (0 a 10 kHz)
idx_f_stft = f_stft <= f_lim_tf

vmax_comum = max(
    np.percentile(amp_stft[idx_f_stft, :], 99.5),
    np.percentile(amp_cwt, 99.5)
)

fig_tf, axs_tf = plt.subplots(
    1, 2, figsize=(7.4, 3.3), sharex=True, sharey=True, constrained_layout=True
)

im_stft = axs_tf[0].pcolormesh(
    t_stft, f_stft[idx_f_stft], amp_stft[idx_f_stft, :],
    shading='gouraud', cmap='viridis', vmin=0.0, vmax=vmax_comum
)
axs_tf[0].set_title("(a)", loc="left", fontsize=10, weight="bold")
axs_tf[0].set_xlabel("Tempo (s)")
axs_tf[0].set_ylabel("Frequência (Hz)")

im_cwt = axs_tf[1].pcolormesh(
    t_cwt, f_cwt, amp_cwt,
    shading='gouraud', cmap='viridis', vmin=0.0, vmax=vmax_comum
)
axs_tf[1].set_title("(b)", loc="left", fontsize=10, weight="bold")
axs_tf[1].set_xlabel("Tempo (s)")

axs_tf[0].set_xlim(0.0, t_lim_tf)
axs_tf[0].set_ylim(100.0, f_lim_tf)  # Elimina a faixa branca inferior em (b)

cbar = fig_tf.colorbar(im_cwt, ax=axs_tf, pad=0.02, extend='max')
cbar.set_label("Amplitude (g)")

plt.show()